In [1]:
from find_SS_inundation_RCP import *
from config import *
import multiprocessing
from tqdm.notebook import tqdm
import os

In [1]:
#ssh to server
#screen -S jupyter
#conda activate
#jupyter start

In [7]:
## %% check all dirs exist
import config as config
for var1 in config.__dict__:
    if var1.find('dir') >0:
        if var1.find('dirs') > 0:
            for dir1 in config.__dict__[var1]:
                if not os.path.exists(dir1):
                    print(f"location not found {dir1}")
        else:
            dir1 = config.__dict__[var1]
            if not os.path.exists(dir1):
                print(f"location not found {dir1}")           

location not found C:/Users/sahas/Downloads/SS_DeltaDTM_2024\DEM\fab-dem\s5hqmjcdj8yo2ibzi9b4ew3sn-new
location not found C:/Users/sahas/Downloads/SS_DeltaDTM_2024\climate\GESLA3.0_ALL
location not found C:/Users/sahas/Downloads/SS_DeltaDTM_2024\Output\samples


In [10]:
#generate masks 
open(os.path.join(log_dir,"mask_gen.log"),'a').write(time.ctime()+" : ")
for dem_dir in ss_dem_dirs:
    ocn_mask = ocean_mask(dem_dir,ss_mask_dir,generate_only=True)
    dem_fn_list = os.listdir(dem_dir)
    dem_fn_list = list(filter(lambda x:x.endswith(ss_dem_end),dem_fn_list))
    '''
    for fn1 in tqdm(dem_fn_list):
        ocn_mask.get_mask(fn1)
        #print('.',end="")
    '''
    with multiprocessing.Pool() as pool:
        pool.map(ocn_mask.get_mask,dem_fn_list)
    


In [2]:
#%% Run inundation
log_msg(f"Starting {project} {DEM_name}",level=1)
# done_files = ["historical_2020", "ssp126_2030", "ssp126_2100", "ssp245_2030", "ssp245_2050", "baseline_1984_2014", "ssp245_2100"]
for region_name,dem_dir in zip(ss_region_names,ss_dem_dirs):
    def process_rcp(rcp_val):
        print("processing ", rcp_val)
        ss_nn_fn        = os.path.join(ss_out_dir,ss_nn_tempelate.format(rcp_val))        
        generate_ss_inundation(rcp_val,
                                ss_nn_fn,
                                region_name,
                                dem_dir
                                ) 
        
    if DEM_name in ["HK_50cm","HK_DTM"]:
        with multiprocessing.Pool() as pool:
            pool.map(process_rcp,ss_rcp_values)
    elif DEM_name in ['copernicus']:
        with multiprocessing.Pool(6) as pool:
            pool.map(process_rcp,ss_rcp_values)
    elif DEM_name in ["FAB"]:
        print("check if running")
        # with multiprocessing.Pool(3) as pool:
        #     pool.map(process_rcp,ss_rcp_values)
        for _rcp_value in ss_rcp_values:
            # if _rcp_value in done_files:
            #     continue
            process_rcp(_rcp_value)
    else:
        for ss_rcp in tqdm(ss_rcp_values):
            process_rcp(ss_rcp)        
log_msg("completed generation",level=1)



Starting SS FAB
check if running
processing  historical_2020_v1
processing  historical_2020_v1 C:/Users/sahas/Downloads/SS_DeltaDTM_2024\Preprocessed\SS_NN\SS_NN_historical_2020_v1.tif


C:\Users\sahas\anaconda3\envs\ML\Lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


.completed generation
